# ▶ EJECUTAR TODO — VERIFAKE (TFM)

**Un solo cuaderno**, en Colab o Kaggle, guardando siempre los datos en **tu Google
Drive**. Encadena el pipeline completo y **lanza la app con su URL pública**.

### Cómo usarlo
1. GPU: Colab → *Entorno de ejecución → T4 GPU*. Kaggle → *Settings → Accelerator → GPU*
   y *Settings → Internet → On*.
2. En la celda 1, edita `REPO_URL`.
3. Ejecuta todo.

### Persistencia en Drive
- **Colab:** monta Drive y escribe directo en él. Nada que configurar.
- **Kaggle:** sincroniza con rclone. Configúralo UNA vez:
  1. En tu ordenador: instala rclone y `rclone config` → remoto **`gdrive`**, tipo
     **`drive`**, login de Google.
  2. `rclone config file` → abre el fichero y copia el bloque `[gdrive] ... token=...`.
  3. Kaggle → *Add-ons → Secrets* → secreto **`rclone_conf`** con ese contenido.
  El notebook extrae el token y reconstruye la config, así que aunque Kaggle pierda
  los saltos de línea, funcionará.


## 1. Preparar el entorno (detecta Colab o Kaggle)

In [ ]:
import os, sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = Path("/content").exists() and not IN_KAGGLE

REPO_URL = "https://github.com/pablo-Tesoro/tfm-deepfake-detection.git"
DRIVE_DIR = "TFM_Deepfake"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = f"/content/drive/MyDrive/{DRIVE_DIR}"
    PROJECT_ROOT = "/content/TFM_Deepfake_Detection"
elif IN_KAGGLE:
    WORKSPACE = f"/kaggle/working/{DRIVE_DIR}"
    PROJECT_ROOT = "/kaggle/working/TFM_Deepfake_Detection"
else:
    WORKSPACE, PROJECT_ROOT = str(Path.cwd() / "workspace"), str(Path.cwd())

os.environ["TFM_WORKSPACE"] = WORKSPACE
os.environ["TFM_PROJECT_ROOT"] = PROJECT_ROOT
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)

if not Path(PROJECT_ROOT).exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull -q

!pip install -q timm grad-cam gradio pyyaml tqdm seaborn
!pip install -q --no-deps facenet-pytorch
if IN_KAGGLE:
    !curl -s https://rclone.org/install.sh | sudo bash > /dev/null 2>&1 || apt-get -qq install -y rclone > /dev/null 2>&1
    !rclone version | head -1
print(f"Entorno: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'local'} | Workspace: {WORKSPACE}")

## 2. Conectar con Drive y traer los datos existentes

- **Colab:** nada que hacer.
- **Kaggle:** reconstruye la config de rclone desde el secreto, prueba la conexión y
  descarga de Drive lo que ya tengas.

In [ ]:
DRIVE_REMOTE = f"gdrive:{DRIVE_DIR}"
RCLONE_CONF = "/kaggle/working/rclone.conf"
DRIVE_OK = False

if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        raw = UserSecretsClient().get_secret("rclone_conf")
        # Extraer SOLO el token {...} y reconstruir un rclone.conf bien formateado
        # (por si el secreto de Kaggle perdió los saltos de línea).
        i, j = raw.find("{"), raw.rfind("}")
        if i == -1 or j == -1:
            raise ValueError("No encuentro el token {...} en el secreto 'rclone_conf'.")
        token = raw[i:j + 1].replace(chr(10), "").replace(chr(13), "").strip()
        conf_clean = "[gdrive]" + chr(10) + "type = drive" + chr(10) + "token = " + token + chr(10)
        Path(RCLONE_CONF).write_text(conf_clean)
        print("Config creada. Claves:", [l.split(" = ")[0] for l in conf_clean.strip().splitlines()])

        print("Probando conexión con Drive...")
        test = get_ipython().getoutput(f"rclone --config {RCLONE_CONF} lsd gdrive: --max-depth 1")
        if any(("Failed" in l) or ("didn't" in l) or ("error" in l.lower()) for l in test):
            raise RuntimeError("rclone no pudo conectar: " + chr(10).join(test))
        print("Conexión OK. Descargando datos existentes de Drive (si los hay)...")
        !rclone --config {RCLONE_CONF} copy {DRIVE_REMOTE} {WORKSPACE} --transfers 8 --checkers 8 --ignore-existing
        DRIVE_OK = True
        print("Sincronización inicial OK.")
    except Exception as e:
        print("[AVISO] Drive no configurado:", e)
        print("Revisa el secreto 'rclone_conf' y que el token no esté revocado.")
        print("Sin Drive, los datos se guardarán solo en /kaggle/working (temporal).")
else:
    DRIVE_OK = True
    print("Colab/local: los datos ya se escriben directamente en el workspace.")

## 3. Ejecutar todo y lanzar la app

Encadena: splits → (descarga opcional) → **vídeo→embeddings fusionados** →
entrenamiento → evaluación → **experimentos** (curva de aprendizaje,
EfficientNet vs ResNet, métricas por método) → app.

> La comparativa de backbones añade, **solo la primera vez**, una pasada
> completa vídeo→embedding para ResNet (luego queda cacheada). Ponla a `None`
> si quieres saltártela.

In [ ]:
sys.path.insert(0, os.environ["TFM_PROJECT_ROOT"])
from run_all import run_pipeline

demo = run_pipeline(
    download=False,               # los vídeos ya están en el Drive
    retrain=True,                 # reentrenar con el dataset completo
    make_figs=True,               # figuras básicas para la memoria
    experiments=True,             # los 3 experimentos avanzados
    compare_backbone="resnet50",  # activa la comparativa EfficientNet vs ResNet
    share=True,                   # app con enlace público
)

## 4. Guardar todo en Drive (persistencia)

In [ ]:
if IN_KAGGLE and DRIVE_OK:
    print("Guardando resultados en Drive (rclone)...")
    !rclone --config {RCLONE_CONF} copy {WORKSPACE} {DRIVE_REMOTE} --transfers 8 --checkers 8
    print("Persistido en Drive: MyDrive/" + DRIVE_DIR)
elif IN_KAGGLE:
    print("Drive no configurado: los datos están solo en /kaggle/working (temporal).")
else:
    print("Colab/local: ya está todo guardado en el workspace.")

## 5. (Opcional) Recuperar el enlace de la app

In [ ]:
print("URL pública:", getattr(demo, "share_url", None))